In [ ]:
# First we have to install the python package
!pip install groupappeals

In [ ]:
# Now we pull the test data from the GitHub Repository
!wget -q https://raw.githubusercontent.com/dpltr22/comptext-2026-group-appeals/main/data/sample_manifestos.csv

In [ ]:
import pandas as pd
from IPython.display import display, HTML
from pathlib import Path
from groupappeals import (extract_entities, detect_stance,​
                          detect_policy, classify_groups,​
                          run_full_pipeline)

Exercise A:

In [ ]:
# ── Sentences to annotate ────────────────────────────
sentences = [
    "We will ensure that workers receive a real living wage and better protections at work.",
    "The number of young people not in education, employment or training has risen sharply.",
    "Our plan will cut taxes for small business owners, helping them invest and create jobs.",
    "Migrants make enormous contributions to our economy, our public services, and our culture.",
    "Criminals who commit violent offences must face the full force of the law.",
]

# ── Your annotations should look like this. Please refer back to the codebook here: https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/FVI3QW ────────────────────────────────
#    group_mentioned : str or None    (exact span, e.g. "workers")
#    stance          : str or None    ("positive", "negative", "neutral")
#    policy          : str or None    ("yes" or "no")

your_annotations = [
    {"group_mentioned": None, "stance": None, "policy": None},   # Sentence 1
    {"group_mentioned": None, "stance": None, "policy": None},   # Sentence 2
    {"group_mentioned": None, "stance": None, "policy": None},   # Sentence 3
    {"group_mentioned": None, "stance": None, "policy": None},   # Sentence 4
    {"group_mentioned": None, "stance": None, "policy": None},   # Sentence 5
]

# ── Build a summary table ────────────────────────────────────────────
rows = []
for i, (sent, ann) in enumerate(zip(sentences, your_annotations), 1):
    rows.append({
        "#": i,
        "Sentence": sent,
        "Group": ann["group_mentioned"] or "❓",
        "Stance": ann["stance"] or "❓",
        "Policy": ann["policy"] or "❓",
    })

df = pd.DataFrame(rows)
df.to_csv("annotations_exercise.csv", index=False)

# Pretty-print with word-wrap for the sentence column
display(HTML(
    df.to_html(index=False, escape=False)
      .replace('<table', '<table style="width:100%; border-collapse:collapse"')
      .replace('<td>', '<td style="padding:8px; border:1px solid #ddd; vertical-align:top">')
      .replace('<th>', '<th style="padding:8px; border:1px solid #ddd; background:#1a5276; color:white; text-align:left">')
))

# Count how many are still unfilled
done = sum(1 for a in your_annotations if a["group_mentioned"] is not None)
print(f"\n✅ {done}/{len(sentences)} annotated — {'all done!' if done == len(sentences) else 'keep going!'}")

Exercise B

In [ ]:
"""
Exercise B — Run the groupappeals pipeline (~20 min)
====================================================

Goal: apply the full pipeline to a small CSV and inspect the output. Compare
the model's group detections with your manual annotations from Exercise A.

Run from the repo root:

Switch to your own data by changing INPUT_FILE.
"""

from pathlib import Path

from groupappeals import run_full_pipeline

# ---------------------------------------------------------------------------
# Configuration — edit these to point at your own data
# ---------------------------------------------------------------------------
INPUT_FILE = "sample_manifestos.csv"
OUTPUT_FILE = "sample_results.csv"


def main() -> None:
    print(f"Reading: {INPUT_FILE}")
    print(f"Writing: {OUTPUT_FILE}\n")

    result_df = run_full_pipeline(
        input_file=str(INPUT_FILE),
        output_file=str(OUTPUT_FILE),
        text_column="text",
        device = "cuda",
        # Build a composite ID like 'Labour_2019_1' for full traceability
        create_composite_id=["party", "year", "sentence_id"],
        clean_labels=True,
        split_groups=True,
        batch_size=8,           # bump to 32 or 64 on a GPU
    )

    print(f"\nDone. {len(result_df)} rows written.")
    print("\nFirst few rows of the output:")
    cols = [
        "text_id", "text", "Exact.Group.Text",
        "Stance_Clean", "Policy_Clean", "Group1",
    ]
    print(result_df[cols].head(10).to_string(index=False))


if __name__ == "__main__":
    main()

Exercise C

In [ ]:
"""
Exercise C — Explore & Validate Results (~15 min)
================================================

Goal: compute simple distributions, spot-check 5–10 rows by hand, and
characterise the kinds of errors the model makes.

Run from the repo root after Exercise B has produced sample_results.csv:

"""

REPO_ROOT = Path(__file__).resolve().parent.parent
RESULTS_FILE = REPO_ROOT / "sample_results.csv"


def main() -> None:
    if not RESULTS_FILE.exists():
        raise SystemExit(
            f"Results not found at {RESULTS_FILE}.\n"
            "Run exercises/exercise_B_pipeline.py first."
        )

    df = pd.read_csv(RESULTS_FILE)

    # 1. How many sentences had a group detected?
    has_group = df["Exact.Group.Text"].notna()
    print(f"Sentences with a detected group: {has_group.sum()} / {len(df)}")

    # 2. Stance distribution
    print("\nStance distribution (rows with a group):")
    print(df.loc[has_group, "Stance_Clean"].value_counts())

    # 3. Policy distribution
    print("\nPolicy distribution (rows with a group):")
    print(df.loc[has_group, "Policy_Clean"].value_counts())

    # 4. Most common group categories
    if "Group1" in df.columns:
        print("\nTop 10 group categories:")
        print(df["Group1"].value_counts().head(10))

    # 5. Sample 10 rows for manual inspection
    print("\nRandom sample of 10 detections — inspect by eye:")
    cols = [
        "text", "Exact.Group.Text",
        "Stance_Clean", "Policy_Clean", "Group1",
    ]
    sample_n = min(10, has_group.sum())
    if sample_n > 0:
        print(df[has_group][cols].sample(sample_n, random_state=42).to_string(index=False))

    # ------------------------------------------------------------------
    # Discussion prompts
    # ------------------------------------------------------------------
    print("""

Discussion prompts
------------------
- Do the detected group spans match what you would have annotated?
- Where does the model under- or over-detect groups?
- Are stance labels plausible? (in manifestos, expect a positive majority)
- Which sentences would you re-check on a larger run?
""")


if __name__ == "__main__":
    main()
